# Centralised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For CLSTM, there is one centralised model, so early stopping is checked once for the global training loop.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from clstm import (
    Datacollector,
    run_centralised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = "E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv" #'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 2
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'clstm'
args_truncated = None
args_SCALING_MODE = 'per_partition'  # 'global' or 'per_partition'
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

dict_ = Datacollector(pd.read_csv(args_path), lags=args_lags, ts=range(15), fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'Scaling mode: {args_SCALING_MODE} | drop_boundary_gap: {args_DROP_BOUNDARY_GAP}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 20.44it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0     Total             0 2003-03-03 00:00:00  12864.0  12389.0  12155.0   
1     Total             0 2003-03-03 01:00:00  12389.0  12155.0  12072.0   
2     Total             0 2003-03-03 02:00:00  12155.0  12072.0  12162.0   
3     Total             0 2003-03-03 03:00:00  12072.0  12162.0  12569.0   
4     Total             0 2003-03-03 04:00:00  12162.0  12569.0  13238.0   

   lags_45  lags_44  lags_43  lags_42  ...   lags_7   lags_6   lags_5  \
0  12072.0  12162.0  12569.0  13238.0  ...  16281.0  16842.0  16503.0   
1  12162.0  12569.0  13238.0  14191.0  ...  16842.0  16503.0  15815.0   
2  12569.0  13238.0  14191.0  15213.0  ...  16503.0  15815.0  14745.0   
3  13238.0  14191.0  15213.0  15646.0  ...  15815.0  14745.0  13444.0   
4  14191.0  15213.0  15646.0  15653.0  ...  14745.0  13444.0  12350.0   

    lags_4   lags_3   lags_2   lags_1        y   post_1   post_2  
0  15815.0  14745.0  13444.0  12350.0

In [3]:
%%time
results = run_centralised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    scaling_mode=args_SCALING_MODE,
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
    checkpoint_path=f'fh{args_fh+1}_cen_lstm.pt',
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Stop epoch:', results['stop_epoch'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
pd.DataFrame(results['round_logs']).tail()

[Centralised] epoch 001/200 | avg_normalised_loss=0.012836 | avg_actual_loss=10830.583627 | delta=nan | stopped=False
[Centralised] epoch 002/200 | avg_normalised_loss=0.009526 | avg_actual_loss=7854.583781 | delta=0.0033099 | stopped=False
[Centralised] epoch 003/200 | avg_normalised_loss=0.008761 | avg_actual_loss=7070.404403 | delta=0.000764939 | stopped=False
[Centralised] epoch 004/200 | avg_normalised_loss=0.007819 | avg_actual_loss=6020.608883 | delta=0.000942132 | stopped=False
[Centralised] epoch 005/200 | avg_normalised_loss=0.007440 | avg_actual_loss=5725.152320 | delta=0.000379267 | stopped=False
[Centralised] epoch 006/200 | avg_normalised_loss=0.006946 | avg_actual_loss=5283.290878 | delta=0.000493449 | stopped=False
[Centralised] epoch 007/200 | avg_normalised_loss=0.007389 | avg_actual_loss=5907.570054 | delta=0.000442248 | stopped=False
[Centralised] epoch 008/200 | avg_normalised_loss=0.006781 | avg_actual_loss=5134.055274 | delta=0.000607564 | stopped=False
[Centrali

,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,stopped
119,120,0.004389,2971.366062,0.000162,False
120,121,0.004414,2941.906008,0.000025,False
121,122,0.004261,2872.112575,0.000153,False
122,123,0.004272,2916.664531,0.000011,False
123,124,0.004280,2863.005715,0.000008,True


In [4]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
6,6,24833,7.800131,0.316341,0.349222
7,7,24833,16.774438,0.157213,0.163971
8,8,24833,13.684434,0.177250,0.183961
9,9,24833,13.320443,0.171010,0.177169
10,Overall,248330,20.252495,0.172760,0.179989


,partition_id,n_test,MAE,MASE,RMSSE
6,6,24833,23.279350,0.944114,0.890544
7,7,24833,99.784432,0.935197,0.924278
8,8,24833,70.843989,0.917619,0.888004
9,9,24833,70.182204,0.901010,0.876564
10,Overall,248330,130.504095,0.920061,0.898401


In [5]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = 'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}GEF_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')
def align_S_for_hierarchicalforecast(S, id_col="unique_id"):
    S = S.copy()
    if id_col not in S.columns:
        S = S.reset_index()
        if id_col not in S.columns:
            S = S.rename(columns={S.columns[0]: id_col})
    bottom_ids = [c for c in S.columns if c != id_col]
    agg_ids = [uid for uid in S[id_col].tolist() if uid not in bottom_ids]
    row_order = agg_ids + bottom_ids
    S_aligned = (
        S
        .set_index(id_col)
        .loc[row_order, bottom_ids]
        .reset_index())
    bottom_block = S_aligned[bottom_ids].tail(len(bottom_ids)).to_numpy()
    if not np.allclose(bottom_block, np.eye(len(bottom_ids))):
        raise ValueError("S Error")
    return S_aligned

S = align_S_for_hierarchicalforecast(S)

reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S_df=S, tags=tags, Y_df=y_df)

recon_results[0].head()

,unique_id,ds,y,clstm,clstm/BottomUp,clstm/MinTrace_method-mint_shrink,clstm/MinTrace_method-wls_var,clstm/MinTrace_method-ols,clstm/MinTrace_method-wls_struct
0,Total,2014-06-30 23:00:00,15138.0,15018.997111,15044.350096,15033.376450,15035.303084,15021.430044,15029.700113
1,Total,2014-07-01 00:00:00,13810.0,13754.281556,13737.468882,13742.434975,13740.260186,13751.342302,13743.892113
2,Total,2014-07-01 01:00:00,12979.0,12958.732940,12918.562679,12933.059642,12928.885043,12953.219997,12937.657999
3,Total,2014-07-01 02:00:00,12468.0,12466.386270,12451.397163,12455.259155,12453.100736,12463.441405,12456.318262
4,Total,2014-07-01 03:00:00,12226.0,12241.138691,12232.431496,12234.196613,12232.757097,12239.153593,12234.608849


In [6]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_cen'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='centralised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh3_cen_forecasts.csv
  per_series_metrics: fh3_cen_per_series_metrics.csv
  metrics_by_level: fh3_cen_metrics_by_level.csv
  overall_metrics: fh3_cen_overall_metrics.csv
  round_logs: fh3_cen_round_logs.csv
  timing: fh3_cen_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,127.244204,0.219951,0.225993,1,centralised
1,1,top,bu,128.930201,0.222866,0.229192,1,centralised
2,1,top,mint_ols,126.912569,0.219378,0.225434,1,centralised
3,1,top,mint_shrinkage,127.071222,0.219652,0.225719,1,centralised
4,1,top,mint_var,127.519895,0.220428,0.226580,1,centralised
5,1,top,naive,1015.797221,1.755882,1.687738,1,centralised
6,1,top,wls_structure,126.864815,0.219295,0.225390,1,centralised
7,2,middle,base,27.223394,0.316564,0.332300,6,centralised
8,2,middle,bu,27.265778,0.316727,0.332454,6,centralised
9,2,middle,mint_ols,27.306741,0.319440,0.331375,6,centralised


In [7]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,37.065011,0.304558,0.318872,10,centralised
1,bu,37.259041,0.304947,0.319284,10,centralised
2,mint_ols,37.042421,0.305716,0.317430,10,centralised
3,mint_shrinkage,36.894505,0.302757,0.316557,10,centralised
4,mint_var,36.977141,0.303524,0.317562,10,centralised
5,naive,250.744261,1.760754,1.686418,10,centralised
6,wls_structure,36.883456,0.302326,0.315426,10,centralised


In [8]:
timing_df

,module,seconds
0,training_sec,7939.969466
1,prediction_sec,23.844176
2,total_sec,7976.228944
